# FedShield CNN — Base Model Training Notebook

Trains the **Multi-Channel CNN** classifier (3/4/5-gram parallel Conv1D) on the
`data_cleaned.csv` dataset. After training, saves weights as `cnn_global_weights.pkl`
so `cnn_server.py` can load them directly.

**Pipeline**
1. Load `data_cleaned.csv` (ham / spam)
2. Generate synthetic **smishing** samples (class 2) via templates + nlpaug augmentation
3. Tokenise with Keras `Tokenizer` (vocab = 10 000)
4. Train / test split (80 / 20, stratified)
5. Multi-Channel CNN training with early stopping
6. Evaluation — accuracy, confusion matrix, classification report
7. Save weights → `cnn_global_weights.pkl`  &  tokeniser → `cnn_tokenizer.json`

In [ ]:
# ── Install dependencies (run once) ─────────────────────────────────────
# Uncomment if needed:
# !pip install tensorflow pandas numpy scikit-learn matplotlib seaborn nlpaug

In [ ]:
import os, re, json, pickle, random, string, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

try:
    import nlpaug.augmenter.char as nac
    NLPAUG_OK = True
    print('nlpaug available ✓')
except ImportError:
    NLPAUG_OK = False
    print('nlpaug not found — augmentation skipped')

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## Step 1 — Configuration

In [ ]:
DATA_FILE      = 'data_cleaned.csv'   # must be in the same folder
WEIGHTS_OUT    = 'cnn_global_weights.pkl'
TOKENIZER_OUT  = 'cnn_tokenizer.json'

VOCAB_SIZE  = 10_000
MAX_LEN     = 300
EMBED_DIM   = 128
NUM_CLASSES = 3
BATCH_SIZE  = 64
EPOCHS      = 15          # early stopping will kick in before this
TEST_SIZE   = 0.20
SEED        = 42

LABEL_MAP = {0: 'ham', 1: 'spam', 2: 'smishing'}
print('Config OK')

## Step 2 — Load CSV data (ham / spam)

In [ ]:
def clean(text):
    t = str(text).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', t).strip().lower()

assert os.path.exists(DATA_FILE), f'{DATA_FILE} not found in {os.getcwd()}'

df = pd.read_csv(DATA_FILE, sep=';', index_col=0)
df = df.dropna(subset=['class', 'message']).reset_index(drop=True)
df['class'] = df['class'].str.strip().str.lower()
df = df[df['class'].isin(['ham', 'spam'])].reset_index(drop=True)

texts  = [clean(t) for t in df['message']]
labels = [0 if c == 'ham' else 1 for c in df['class']]

print(f'Loaded {len(df):,} rows')
print(df['class'].value_counts())

## Step 3 — Generate synthetic smishing samples (class 2)

In [ ]:
SMISH_TMPL = [
    'urgent your {bank} account has been compromised verify at http://secure-{r}.com login now',
    'your {bank} account will be suspended click http://verify-{r}.net to confirm identity',
    'security alert unusual login detected confirm at http://bank-{r}.com secure',
    'dear customer your account is at risk verify now http://login-{r}.org account',
    'final notice confirm your {bank} details or account closed http://{r}-secure.com',
    'your package could not be delivered update address http://delivery-{r}.com track',
    'tax refund you are eligible for {amt} refund claim at http://irs-{r}.net now',
    'action required verify your paypal account http://paypal-verify-{r}.com',
    'your upi is suspended confirm details at http://upi-{r}.in verify within 24h',
    'sbi alert debit card blocked reactivate http://sbi-{r}.com card activate',
    'hdfc bank account frozen login http://hdfc-{r}.net secure immediately',
    'selected for kyc update visit http://{r}-kyc.com immediately to avoid suspension',
    'alert netbanking temporarily blocked verify at http://{r}-netbank.com now',
    'dear user your {bank} credit card blocked call or click http://{r}.xyz secure',
    'your otp has expired re-login at http://{r}-otp.net to restore access now',
]

BANKS = ['SBI', 'HDFC', 'ICICI', 'Axis', 'PNB', 'Kotak', 'BOI', 'PayTM', 'IDBI']

def gen_smishing(n=500):
    out = []
    for _ in range(n):
        tmpl = random.choice(SMISH_TMPL)
        r    = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
        s    = tmpl.format(r=r, bank=random.choice(BANKS),
                           amt=f'${random.randint(100, 9999)}')
        out.append(s)
    return out

smish_base = gen_smishing(500)

# Also augment existing spam texts as additional smishing samples
spam_texts = [t for t, l in zip(texts, labels) if l == 1]

def augment(text_list, p=0.15):
    if not NLPAUG_OK or not text_list:
        return text_list
    aug = nac.RandomCharAug(action='substitute', aug_char_p=p)
    return [aug.augment(t)[0] for t in text_list]

aug_smish = augment(spam_texts[:200], p=0.20)

all_smish = smish_base + aug_smish
texts  += all_smish
labels += [2] * len(all_smish)

ham_n   = labels.count(0)
spam_n  = labels.count(1)
smish_n = labels.count(2)
print(f'Dataset → ham:{ham_n}  spam:{spam_n}  smishing:{smish_n}  total:{len(texts)}')

### Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
counts = [ham_n, spam_n, smish_n]
colors = ['#00e5a0', '#ff4f4f', '#ff9500']
ax.bar(['Ham', 'Spam', 'Smishing'], counts, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(counts):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## Step 4 — Tokenise & pad sequences

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)

seqs = tokenizer.texts_to_sequences(texts)
X    = pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')
y    = np.array(labels, dtype=np.int32)

# Save tokenizer for cnn_server.py
with open(TOKENIZER_OUT, 'w') as f:
    f.write(tokenizer.to_json())

print(f'Vocabulary size : {len(tokenizer.word_index):,}')
print(f'X shape         : {X.shape}')
print(f'y distribution  : {dict(zip(*np.unique(y, return_counts=True)))}') 
print(f'Tokenizer saved → {TOKENIZER_OUT}')

## Step 5 — Train / test split (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

print(f'Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}')
print(f'Train class dist: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test  class dist: {dict(zip(*np.unique(y_test,  return_counts=True)))}')

## Step 6 — Multi-Channel CNN model

Three parallel Conv1D streams capture **trigram**, **4-gram**, and **5-gram** patterns.
Their pooled outputs are concatenated before the classification head.

In [ ]:
def create_cnn_model():
    inp = layers.Input(shape=(MAX_LEN,), name='token_ids')
    x   = layers.Embedding(VOCAB_SIZE, EMBED_DIM, name='embedding')(inp)

    # Parallel convolutional channels
    p3 = layers.GlobalMaxPooling1D(name='pool3')(
             layers.Conv1D(128, 3, activation='relu', padding='same', name='conv3')(x))
    p4 = layers.GlobalMaxPooling1D(name='pool4')(
             layers.Conv1D(128, 4, activation='relu', padding='same', name='conv4')(x))
    p5 = layers.GlobalMaxPooling1D(name='pool5')(
             layers.Conv1D(128, 5, activation='relu', padding='same', name='conv5')(x))

    merged = layers.Concatenate(name='concat')([p3, p4, p5])
    d      = layers.Dense(64, activation='relu', name='dense_64')(merged)
    drop   = layers.Dropout(0.5, name='dropout')(d)
    out    = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(drop)

    model  = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = create_cnn_model()
model.summary()

## Step 7 — Training

In [ ]:
cbs = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=3,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=2, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cbs,
    verbose=1,
)

### Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train Acc',  color='#4d8bff')
ax1.plot(history.history['val_accuracy'], label='Val Acc',    color='#00e5a0')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')

ax2.plot(history.history['loss'],     label='Train Loss', color='#ff4f4f')
ax2.plot(history.history['val_loss'], label='Val Loss',   color='#ff9500')
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch')

plt.suptitle('Multi-Channel CNN Training', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8 — Evaluation

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {acc*100:.2f}%')

In [ ]:
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print('\nClassification Report:')
print(classification_report(y_test, y_pred,
      target_names=['Ham', 'Spam', 'Smishing']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham','Spam','Smishing'],
            yticklabels=['Ham','Spam','Smishing'],
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Multi-Channel CNN', fontweight='bold')
plt.tight_layout(); plt.show()

## Step 9 — Save weights for `cnn_server.py`

Saves the trained weights as `cnn_global_weights.pkl`.
When `cnn_server.py` starts, it loads this file automatically.

In [ ]:
weights = [w.astype('float32') for w in model.get_weights()]

with open(WEIGHTS_OUT, 'wb') as f:
    pickle.dump(weights, f)

size_mb = os.path.getsize(WEIGHTS_OUT) / 1024 / 1024
print(f'Weights saved  → {WEIGHTS_OUT}  ({size_mb:.2f} MB)')
print(f'Tokenizer saved→ {TOKENIZER_OUT}')
print()
print('Next step: run  python cnn_server.py')
print('It will load these weights automatically on startup.')

## Step 10 — Quick sanity check

In [ ]:
SAMPLES = [
    ('Free entry into our prize draw! Call now to claim your reward!',    'spam'),
    ('Hey, are you free for lunch tomorrow?',                             'ham'),
    ('URGENT: Your HDFC account suspended. Verify http://hdfc-abc.com',   'smishing'),
    ('Your package could not be delivered. Click http://track-xyz.net',   'smishing'),
    ('Ok I will call you back in 5 minutes',                              'ham'),
    ('Congratulations! You won a FREE holiday. Text WIN to 88800',        'spam'),
]

print(f'{"Text":<55} {"True":<10} {"Predicted":<12} {"Conf"}')
print('-' * 90)
for text, true_label in SAMPLES:
    seq  = tokenizer.texts_to_sequences([clean(text)])
    x_in = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    prob = model.predict(x_in, verbose=0)[0]
    pred = LABEL_MAP[int(np.argmax(prob))]
    conf = f'{prob.max()*100:.1f}%'
    flag = '✓' if pred == true_label else '✗'
    print(f'{flag} {text[:53]:<53} {true_label:<10} {pred:<12} {conf}')